# Donut Training Diagnostics

Loads a checkpoint (or the base model) and inspects what goes in and what comes out.

In [1]:
import re
import sys


from pathlib import Path

import numpy as np
import torch
from dataset import DonutDataset, build_processor, load_samples
from PIL import Image
from transformers import VisionEncoderDecoderModel

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME = "naver-clova-ix/donut-base"
CKPT_PATH = None  # e.g. 'checkpoints/epoch_001_val0.1234.pt'
DATA_JSON = "../test_data/train.json"
TOKEN2JSON_FORMAT = False  # must match what was used during training
MAX_NEW_TOKENS = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/home/fotis/projects/inference/donut_train/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
samples = load_samples("../test_data/train.json")
samples[0]

{'image': PosixPath('../test_data/images/train/fake_01.jpg'),
 'fields': [{'field_name': 'BR/COMMISSION_PAYEMENT/E-mail',
   'annotator_text': 'alice@example.com'},
  {'field_name': 'BR/COMMISSION_PAYEMENT/data_emissao',
   'annotator_text': '15/03/2024'},
  {'field_name': 'BR/COMMISSION_PAYEMENT/valor_da_nota',
   'annotator_text': '1250.00'}]}

In [3]:
d = DonutDataset(
    samples,
    build_processor("naver-clova-ix/donut-base"),
    token2json_format=True,
    include_missing=True,
)

In [5]:
d[0]

{'pixel_values': tensor([[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]],
 
         [[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]],
 
         [[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]]),
 'labels': tensor([57525, 41040, 46192, 41403,  2835, 43862, 41072

## 1. Load model + processor

In [4]:
processor = build_processor(MODEL_NAME, TOKEN2JSON_FORMAT)

model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.pad_token_id

if CKPT_PATH:
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    print(
        f"Loaded checkpoint: {CKPT_PATH}  (epoch {ckpt['epoch']}, val_loss {ckpt['val_loss']:.4f})"
    )
else:
    print("No checkpoint — using base pretrained weights")

model = model.to(DEVICE).eval()
tok = processor.tokenizer
print(f"Vocab size: {len(tok)}")

KeyboardInterrupt: 

## 2. Load a sample

In [ ]:
from IPython.display import display

samples = load_samples(Path(IMAGES_DIR), Path(ANNOTATIONS_DIR))
print(f"Loaded {len(samples)} samples")

ds = DonutDataset(samples, processor, token2json_format=TOKEN2JSON_FORMAT)

SAMPLE_IDX = 0
item = ds[SAMPLE_IDX]
sample = samples[SAMPLE_IDX]

print(f"\nSample : {Path(sample['image']).name}")
print(f"Fields : {[f['field_name'].split('/')[-1] for f in sample['fields']]}")
print(f"Target : {item['target_text']!r}")

raw_image = Image.open(sample["image"]).convert("RGB")
display(raw_image.resize((320, int(320 * raw_image.height / raw_image.width))))

## 3. Token ID sanity checks

In [ ]:
start_id = model.config.decoder_start_token_id
pad_id = model.config.pad_token_id
vocab_size = len(tok)

print(f"Vocab size               : {vocab_size}")
print(f'decoder_start_token_id   : {start_id}  → "{tok.decode([start_id])}"')
print(f'pad_token_id             : {pad_id}    → "{tok.decode([pad_id])}"')
print(f'eos_token_id             : {tok.eos_token_id}  → "{tok.eos_token}"')
print(f"Random-chance loss (nats): {np.log(vocab_size):.3f}")
print()

from dataset import ALL_SPECIAL_TOKENS, ALL_SPECIAL_TOKENS_T2J

tokens_to_check = ALL_SPECIAL_TOKENS_T2J if TOKEN2JSON_FORMAT else ALL_SPECIAL_TOKENS
print("Additional token IDs:")
for t in tokens_to_check:
    tid = tok.convert_tokens_to_ids(t)
    ok = tid != tok.unk_token_id
    pieces = tok(t, add_special_tokens=False).input_ids
    print(
        f"  {t:40s} → {tid}  {'OK' if ok else 'MISSING — maps to UNK!'}  ({len(pieces)} piece{'s' if len(pieces) != 1 else ''})"
    )

## 4. Pixel value stats

In [ ]:
pv = item["pixel_values"].unsqueeze(0)  # (1, 3, H, W)
print(f"pixel_values shape : {tuple(pv.shape)}")
print(f"dtype              : {pv.dtype}")
print(f"min / max / mean   : {pv.min():.4f} / {pv.max():.4f} / {pv.mean():.4f}")
if pv.min() >= -1.1 and pv.max() <= 1.1:
    print("Range looks correct (roughly [-1, 1])")
else:
    print("WARNING: range outside [-1, 1] — check normalization")

## 5. Label decoding round-trip

In [ ]:
labels = item["labels"]

unmasked = labels.clone()
unmasked[unmasked == -100] = tok.pad_token_id
active_ids = unmasked[unmasked != tok.pad_token_id]

print(f"Non-pad label tokens : {len(active_ids)}")
print(f"Decoded              : {tok.decode(active_ids)!r}")
print()
ends_with_eos = len(active_ids) > 0 and active_ids[-1].item() == tok.eos_token_id
print(
    f"Ends with EOS        : {ends_with_eos} {'✓' if ends_with_eos else '✗ labels missing EOS!'}"
)

pieces = tok.convert_ids_to_tokens(active_ids.tolist())
print(f"\nToken pieces: {pieces}")

## 6. Teacher-forced forward pass

In [ ]:
pv_dev = pv.to(DEVICE)
lbl_dev = labels.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    out = model(pixel_values=pv_dev, labels=lbl_dev)

print(f"Teacher-forced loss : {out.loss.item():.4f}")
print(f"(Random-chance      : {np.log(len(tok)):.3f})")
print()

logits = out.logits[0]
pred_ids = logits.argmax(dim=-1)
true_ids = labels[: logits.shape[0]]

active = true_ids != -100
pred_active = pred_ids[active]
true_active = true_ids[active]

print(f"{'Pos':>4}  {'True token':<28} {'Predicted':<28}")
print("─" * 64)
for i, (t, p) in enumerate(zip(true_active.tolist(), pred_active.tolist())):
    tt = tok.decode([t])
    pt = tok.decode([p])
    match = "✓" if t == p else "✗"
    print(f"{i:4d}  {tt:<28} {pt:<28} {match}")
    if i >= 30:
        print(f"      … ({len(true_active) - 31} more positions)")
        break

## 7. Greedy generation (no teacher forcing)

In [ ]:
pv_dev = pv.to(DEVICE)

with torch.no_grad():
    gen_ids = model.generate(
        pixel_values=pv_dev,
        decoder_start_token_id=model.config.decoder_start_token_id,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        bad_words_ids=[[tok.unk_token_id]],
    )

sequence = processor.batch_decode(gen_ids)[0]
sequence = sequence.replace(tok.eos_token, "").replace(tok.pad_token, "")
sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()  # strip task start token

print("Raw generated tokens:")
print(repr(processor.batch_decode(gen_ids)[0]))
print()
print("Post-processed sequence:")
print(repr(sequence))
print()
print("Target:")
print(repr(item["target_text"]))

## 8. Parse to structured dict (token2json)

Only meaningful when `TOKEN2JSON_FORMAT = True`.

In [ ]:
if TOKEN2JSON_FORMAT:
    parsed = processor.token2json(sequence)
    print("Parsed output:")
    import json

    print(json.dumps(parsed, indent=2, ensure_ascii=False))
else:
    print("TOKEN2JSON_FORMAT=False — parsing the sequence with a simple regex instead.")
    import re as _re

    # matches <field> value <field> in the legacy format
    parsed = {}
    for m in _re.finditer(r"<(\w[\w-]*)>\s*(.*?)\s*<\1>", sequence):
        parsed[m.group(1)] = m.group(2)
    import json

    print(json.dumps(parsed, indent=2, ensure_ascii=False))